# Using the public synthetic-derm dataset

The 2023 paper released 458,920 synthetic images; the 2024 Hugging Face release grew that to
about one million (600 GB) in [WebDataset](https://github.com/webdataset/webdataset) format,
split into TAR shards of at most 5,000 images so that subsets can be pulled without
downloading everything.

Two versions exist:

1. [`tbuckley/synthetic-derm-1M`](https://huggingface.co/datasets/tbuckley/synthetic-derm-1M):
   per-label shards, `shard-{label}-{method}-{submethod}-{index}.tar`, for fine-grained retrieval.
2. [`tbuckley/synthetic-derm-1M-train`](https://huggingface.co/datasets/tbuckley/synthetic-derm-1M-train):
   shuffled shards grouped by generation method, `shard-{method}-{index}.tar`, for training.

`method` is `finetune-text-to-image`, `finetune-inpaint`, `pretrained-text-to-image` or
`pretrained-inpaint`; `submethod` is `text-to-image`, `inpaint` or `inpaint-outpaint`. Every
sample has a PNG and a JSON with `md5hash` (the Fitzpatrick 17k image it was generated from),
`label`, `method`, `submethod` and `generation_num`.

Shards generated with the 2026 backends will sit next to these under
`shard-{label}-{generator}-{method}-{index}.tar` (see `huggingface/compile_webdataset_v2.py`).

In [ ]:
CACHE_DIR = "/workspace/.cache/wds"
LABELS = ["basal-cell-carcinoma", "allergic-contact-dermatitis", "lupus-erythematosus"]

In [ ]:
import io, json, os, re
import matplotlib.pyplot as plt
import pandas as pd
import webdataset as wds
from huggingface_hub import HfApi
from PIL import Image
os.makedirs(CACHE_DIR, exist_ok=True)

## Dataset statistics

The repository ships a crawl of the full dataset in `huggingface/folder_counts.csv`.

In [ ]:
from pathlib import Path
repo_root = Path(os.environ.get("SYNDERM_REPO", ".")).resolve()
counts = repo_root / "huggingface" / "folder_counts.csv"
if not counts.exists():
    counts = Path("/root/synthetic-derm/huggingface/folder_counts.csv")
df = pd.read_csv(counts).rename(columns={"Image Count": "count", "Folder": "folder"})
df["label"] = df["folder"].str.split("/").str[-1]
df["method"] = df["folder"].str.split("/").str[0]
print("total images:", df["count"].sum())
print(df.groupby("method")["count"].sum().sort_values(ascending=False))
df.groupby("label")["count"].sum().sort_values(ascending=False).head(15)

## Listing shards

Shard names are the index: query the Hub for the files and filter by label and method.

In [ ]:
api = HfApi()
files = [f for f in api.list_repo_files("tbuckley/synthetic-derm-1M", repo_type="dataset") if f.endswith(".tar")]
print(len(files), "shards; first few:"); print("\n".join(files[:5]))
pattern = re.compile(r"data/shard-(?P<label>{})-finetune-text-to-image-text-to-image-(?P<index>\d{{5}})\.tar".format("|".join(LABELS)))
selected = sorted(f for f in files if pattern.match(f))
print(len(selected), "selected shards for", LABELS)

## Streaming a subset with WebDataset

Shards are streamed over HTTPS through the Hub's `resolve` URLs; `cache_dir` keeps a local
copy. WebDataset is sequential; for random access, use `synderm.data.load_2024_shards`, which
unpacks a bounded number of samples into the synderm folder layout.

In [ ]:
urls = [f"https://huggingface.co/datasets/tbuckley/synthetic-derm-1M/resolve/main/{f}" for f in selected[:len(LABELS)]]
dataset = (wds.WebDataset(urls, shardshuffle=False, cache_dir=CACHE_DIR)
           .decode("pil").to_tuple("png", "json"))
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for ax, (image, meta) in zip(axes, dataset):
    ax.imshow(image); ax.axis("off"); ax.set_title(f"{meta['label']}\n{meta['submethod']}", fontsize=8)
plt.show()
print("metadata of the last sample:", meta)

## Random access through synderm

`load_2024_shards` restricts to given labels, optionally to images generated from a set of
real image hashes, and returns a `SyntheticDataset` whose `ref_id` joins back to Fitzpatrick 17k.

In [ ]:
from synderm.data import load_2024_shards
ds = load_2024_shards(LABELS[:1], method="finetune-text-to-image", submethod="text-to-image", max_per_label=20)
item = ds[0]
print(len(ds), "samples;", {k: v for k, v in item.items() if k not in ("image", "original")})

## Mixing shards for training

WebDataset's `RandomMix` interleaves several streams with sampling weights, which is how the
classifier vignettes mix real and synthetic images at a chosen ratio.

In [ ]:
from webdataset import RandomMix
streams = [wds.WebDataset([u], shardshuffle=False, cache_dir=CACHE_DIR).decode("pil").to_tuple("png", "json") for u in urls]
mixed = RandomMix(streams, [1.0] * len(streams))
from collections import Counter
c = Counter(meta["label"] for i, (_, meta) in zip(range(60), mixed))
c